# Model 2: Embedding + BiLSTM

This notebook builds the neural model for the Sentiment140 project. It uses trainable word embeddings and a bidirectional LSTM to read each tweet in both directions before predicting negative or positive sentiment.

The goal is to test whether a small neural model can capture word order and context better than the classical TF-IDF baseline while staying much smaller than a transformer.

In [ ]:
import re
import os
import time
import random
import joblib

from collections import Counter
    
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset, concatenate_datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

## Reproducibility and device setup

The random seed is fixed so that sampling, initialization, and training are easier to reproduce. The notebook uses a GPU when available and falls back to CPU otherwise.

In [54]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [55]:
SUBSAMPLE_SIZE = 50_000

## Load Sentiment140

Sentiment140 contains 1.6 million English tweets labeled as negative or positive. The original positive label is `4`, which is later remapped to `1`.

In [56]:
ds = load_dataset("contemmcm/sentiment140")

data = ds["complete"]

data[0]

{'text': "Just back home from a little gathering with some old friends.. It was really fun, they're still the same. ",
 'label': 1}

## Create a balanced 50,000-example subset

The project uses 25,000 negative tweets and 25,000 positive tweets to fit available resources while keeping the class distribution balanced.

In [57]:
negative = data.filter(lambda x: x["label"] == 0)
positive = data.filter(lambda x: x["label"] == 1)

n_per_class = SUBSAMPLE_SIZE // 2

negative_sample = negative.shuffle(seed=SEED).select(range(n_per_class))
positive_sample = positive.shuffle(seed=SEED).select(range(n_per_class))

balanced_data = concatenate_datasets([negative_sample, positive_sample]).shuffle(seed=SEED)

print("Balanced sample size:", len(balanced_data))

Balanced sample size: 50000


In [58]:
positive

Dataset({
    features: ['text', 'label'],
    num_rows: 800000
})

## Convert the dataset into a DataFrame

The DataFrame keeps the tweet text and binary labels together, making it easier to clean the text and split the data.

In [59]:
df = pd.DataFrame({
    "text": balanced_data["text"],
    "label": [0 if y == 0 else 1 for y in balanced_data["label"]]
})

df.head()

,text,label
0,I'm actually pretty sad the Duke has left us ...,0
1,"@Heatherrrr_ haha spazzz ;) and i know, i've o...",0
2,@christina1986 what cam did you get now? the ...,1
3,ha! one line css grids framework. http://bit....,1
4,"GOODMORNING... Ugh, twitter fully kicked me of...",0


In [60]:
df["label"].value_counts()

,count
label,
0,25000
1,25000


## Clean the tweet text

URLs are removed, user mentions are normalized to `@user`, hashtags are preserved because they can carry sentiment, and whitespace is cleaned.

In [61]:
def clean_tweet(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Replace @usernames with @user
    text = re.sub(r"@\w+", "@user", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["clean_text"] = df["text"].apply(clean_tweet)

df[["text", "clean_text", "label"]].head(10)

,text,clean_text,label
0,I'm actually pretty sad the Duke has left us ...,I'm actually pretty sad the Duke has left us W...,0
1,"@Heatherrrr_ haha spazzz ;) and i know, i've o...","@user haha spazzz ;) and i know, i've only jus...",0
2,@christina1986 what cam did you get now? the ...,@user what cam did you get now? the one that i...,1
3,ha! one line css grids framework. http://bit....,ha! one line css grids framework. (via @user) ...,1
4,"GOODMORNING... Ugh, twitter fully kicked me of...","GOODMORNING... Ugh, twitter fully kicked me of...",0
5,Itchy eyes,Itchy eyes,0
6,off to school tom. which is soo not good. i ha...,off to school tom. which is soo not good. i ha...,0
7,"@cdncamel LOL, Claire. Even if you pass on th...","@user LOL, Claire. Even if you pass on the roo...",1
8,Really feel like crap this morning feel like ...,Really feel like crap this morning feel like I...,0
9,@peterloggins hehe i know! Luka is the BEST fr...,@user hehe i know! Luka is the BEST friend ever!,1


## Split into train and test sets

The same 80/20 split strategy is used across all models, giving 40,000 training tweets and 10,000 test tweets.

In [62]:
X = df["clean_text"].tolist()
y = df["label"].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 40000
Test size: 10000


## Tokenize tweets

The BiLSTM needs token IDs instead of raw strings. This tokenizer keeps words, mentions, hashtags, numbers, and punctuation-like tokens that may contain sentiment cues.

In [63]:
def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"@\w+|#\w+|[a-zA-Z]+|[0-9]+|[^\s]", text)
    return tokens

print(tokenize("Thanks @user! This project is actually fun :) #nlp"))

['thanks', '@user', '!', 'this', 'project', 'is', 'actually', 'fun', ':', ')', '#nlp']


## Build the vocabulary

The vocabulary is learned from the training data only. Rare words are filtered out to keep the embedding matrix compact.

In [64]:
MAX_VOCAB_SIZE = 50_000
MIN_FREQ = 2

counter = Counter()

for text in X_train:
    counter.update(tokenize(text))

special_tokens = ["<pad>", "<unk>"]

vocab_words = [
    word for word, freq in counter.most_common(MAX_VOCAB_SIZE)
    if freq >= MIN_FREQ
]

itos = special_tokens + vocab_words
stoi = {word: idx for idx, word in enumerate(itos)}

PAD_IDX = stoi["<pad>"]
UNK_IDX = stoi["<unk>"]

print("Vocab size:", len(itos))
print("PAD index:", PAD_IDX)
print("UNK index:", UNK_IDX)

Vocab size: 13031
PAD index: 0
UNK index: 1


## Encode and pad sequences

Each tweet is converted into token IDs and padded or truncated to a fixed length so that batches can be processed efficiently.

In [65]:
MAX_LEN = 50

def encode_text(text, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [stoi.get(token, UNK_IDX) for token in tokens]

    if len(ids) > max_len:
        ids = ids[:max_len]

    attention_length = len(ids)

    while len(ids) < max_len:
        ids.append(PAD_IDX)

    return ids, attention_length

## Prepare PyTorch datasets

The dataset wrapper returns encoded tweets, labels, and sequence lengths for the DataLoader.

In [66]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_ids, length = encode_text(self.texts[idx])

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "length": torch.tensor(length, dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [67]:
BATCH_SIZE = 128

train_dataset = TweetDataset(X_train, y_train)
test_dataset = TweetDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["length"].shape)
print(batch["label"].shape)

torch.Size([128, 50])
torch.Size([128])
torch.Size([128])


## Define the BiLSTM classifier

The model learns embeddings from scratch, reads the tweet forward and backward with an LSTM, applies dropout, and predicts the sentiment class with a linear layer.

In [ ]:
class BiLSTMSentiment(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        output_dim,
        pad_idx,
        num_layers=1,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        output, (hidden, cell) = self.lstm(embedded)

        # Masked mean-pool over the real (non-pad) tokens.
        # NOTE: reading the last-timestep hidden state instead would break a
        # *unidirectional* LSTM -- sequences are right-padded to MAX_LEN, so its
        # final state gets washed out by the trailing <pad>. Mean-pooling is
        # correct for both directions and matches what app.py expects.
        mask = (input_ids != 0).unsqueeze(-1).float()
        pooled = (output * mask).sum(1) / mask.sum(1).clamp(min=1)

        logits = self.fc(self.dropout(pooled))
        return logits

In [69]:
VOCAB_SIZE = len(itos)
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 2

model = BiLSTMSentiment(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    pad_idx=PAD_IDX
).to(device)

model

BiLSTMSentiment(
  (embedding): Embedding(13031, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=2, bias=True)
)

## Configure optimization

Cross-entropy loss is used for binary classification with two output classes, and Adam updates the model parameters.

In [70]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

## Define training and evaluation loops

These helper functions collect loss, accuracy, F1, predictions, and labels so the neural model can be compared with the other approaches.

In [71]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return avg_loss, accuracy, f1

In [72]:
def evaluate(model, dataloader, criterion):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return avg_loss, accuracy, f1, all_preds, all_labels

## Train the model

Training time is recorded because the final comparison includes both model quality and cost.

In [73]:
EPOCHS = 3

start_time = time.time()

history = []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_loss, test_acc, test_f1, _, _ = evaluate(
        model,
        test_loader,
        criterion
    )

    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_f1": train_f1,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_f1": test_f1
    }

    history.append(epoch_result)

    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Test loss:  {test_loss:.4f} | Test acc:  {test_acc:.4f} | Test F1:  {test_f1:.4f}")

train_time = time.time() - start_time

print(f"\nTotal training time: {train_time:.2f} seconds")


Epoch 1/3


Training:   0%|          | 0/313 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/79 [00:00<?, ?it/s]

Train loss: 0.5973 | Train acc: 0.6803 | Train F1: 0.6795
Test loss:  0.5510 | Test acc:  0.7333 | Test F1:  0.7256

Epoch 2/3


Training:   0%|          | 0/313 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/79 [00:00<?, ?it/s]

Train loss: 0.5034 | Train acc: 0.7615 | Train F1: 0.7594
Test loss:  0.5229 | Test acc:  0.7569 | Test F1:  0.7714

Epoch 3/3


Training:   0%|          | 0/313 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/79 [00:00<?, ?it/s]

Train loss: 0.4458 | Train acc: 0.7985 | Train F1: 0.7964
Test loss:  0.4917 | Test acc:  0.7708 | Test F1:  0.7650

Total training time: 11.81 seconds


In [74]:
history_df = pd.DataFrame(history)

history_df

,epoch,train_loss,train_accuracy,train_f1,test_loss,test_accuracy,test_f1
0,1,0.597333,0.680275,0.679498,0.551041,0.7333,0.725646
1,2,0.503403,0.761525,0.759353,0.522853,0.7569,0.771415
2,3,0.445770,0.798525,0.796372,0.491695,0.7708,0.765019


## Evaluate on the test set

The held-out test set measures how well the BiLSTM generalizes beyond the training tweets.

In [75]:
test_loss, test_accuracy, test_f1, y_pred, y_true = evaluate(
    model,
    test_loader,
    criterion
)

print(f"Final test accuracy: {test_accuracy:.4f}")
print(f"Final test F1: {test_f1:.4f}")

Evaluating:   0%|          | 0/79 [00:00<?, ?it/s]

Final test accuracy: 0.7708
Final test F1: 0.7650


In [76]:
print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"]
))

              precision    recall  f1-score   support

    negative       0.76      0.80      0.78      5000
    positive       0.78      0.75      0.77      5000

    accuracy                           0.77     10000
   macro avg       0.77      0.77      0.77     10000
weighted avg       0.77      0.77      0.77     10000



## Measure inference latency

Latency shows how fast the model can respond in an interactive setting such as the Y demo.

In [ ]:
# Single-example latency over the first 1000 test tweets, measured ONE tweet at
# a time so it is comparable to the classical and DistilBERT models. (Originally
# this ran a single batched forward over all of X_test and divided by N, which
# reports batched throughput, not per-example latency.)
model.eval()
sample_texts = X_test[:1000]

start_time = time.time()
with torch.no_grad():
    for text in sample_texts:
        ids = encode_text(text)[0]
        input_tensor = torch.tensor([ids], dtype=torch.long).to(device)
        _ = model(input_tensor)
total_inference_time = time.time() - start_time
latency_per_example_ms = (total_inference_time / len(sample_texts)) * 1000

print(f"Total inference time: {total_inference_time:.4f} seconds")
print(f"Inference latency: {latency_per_example_ms:.4f} ms/example")

## Save the trained model artifacts

The model, vocabulary, and configuration are saved so the interface can load the BiLSTM without rerunning the notebook.

In [ ]:
# Save in the format app.py loads:  models/bilstm.pt = {'state', 'vocab'}
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "app.py").exists() and (ROOT.parent / "app.py").exists():
    ROOT = ROOT.parent
MODELS_DIR = ROOT / "models"; MODELS_DIR.mkdir(exist_ok=True)

torch.save({"state": model.state_dict(), "vocab": stoi}, MODELS_DIR / "bilstm.pt")
print("Saved BiLSTM ->", MODELS_DIR / "bilstm.pt")

In [79]:
model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
vocab_size_mb = os.path.getsize(vocab_path) / (1024 * 1024)
config_size_mb = os.path.getsize(config_path) / (1024 * 1024)

total_size_mb = model_size_mb + vocab_size_mb + config_size_mb

print(f"Model size: {model_size_mb:.2f} MB")
print(f"Vocab size: {vocab_size_mb:.2f} MB")
print(f"Config size: {config_size_mb:.2f} MB")
print(f"Total saved size: {total_size_mb:.2f} MB")

Model size: 5.87 MB
Vocab size: 0.21 MB
Config size: 0.00 MB
Total saved size: 6.08 MB


## Summarize neural model results

The final result dictionary uses the same fields as the other notebooks, making the model comparison consistent.

In [80]:
neural_results = {
    "Model": "Embedding + BiLSTM",
    "Dataset Size": len(df),
    "Train Size": len(X_train),
    "Test Size": len(X_test),
    "Accuracy": test_accuracy,
    "F1": test_f1,
    "Train Time (s)": train_time,
    "Inference (ms/example)": latency_per_example_ms,
    "Model Size (MB)": total_size_mb
}

neural_results_df = pd.DataFrame([neural_results])

neural_results_df

,Model,Dataset Size,Train Size,Test Size,Accuracy,F1,Train Time (s),Inference (ms/example),Model Size (MB)
0,Embedding + BiLSTM,50000,40000,10000,0.7708,0.765019,11.808705,0.000414,6.084473


## Save results for the final comparison

The CSV output can be combined with the classical and DistilBERT results for the presentation table.

In [81]:
neural_results_df.to_csv("neural_model_results.csv", index=False)

print("Saved results to neural_model_results.csv")

Saved results to neural_model_results.csv
